In [1]:
%pip install -q unsloth

Note: you may need to restart the kernel to use updated packages.


In [2]:
import os

# Prevent Unsloth's Qwen3 attention tensors from being split
# across Kaggle's two T4 GPUs.
os.environ["CUDA_VISIBLE_DEVICES"] = "0"

print("Single-GPU mode configured")

Single-GPU mode configured


In [3]:
from importlib.metadata import version

import torch
import unsloth
from unsloth import FastLanguageModel

print("Unsloth version:", version("unsloth"))
print("PyTorch version:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())
print("GPU:", torch.cuda.get_device_name(0))
print("Visible GPU count:", torch.cuda.device_count())
print("FastLanguageModel import: PASSED")

🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.
🦥 Unsloth Zoo will now patch everything to make training faster!
Unsloth version: 2026.9.4
PyTorch version: 2.10.0+cu128
CUDA available: True
GPU: Tesla T4
Visible GPU count: 1
FastLanguageModel import: PASSED


## 3. Load and Audit the Spider Text-to-SQL Dataset

This section loads the Spider benchmark and verifies its splits, columns, missing values, database coverage, and potential database leakage.

In [4]:
from datasets import load_dataset

DATASET_NAME = "hujudev/spider-text-2-sql"

spider = load_dataset(DATASET_NAME)

print("Dataset loaded successfully")
print(spider)

for split_name, split_data in spider.items():
    print(f"\n{'=' * 60}")
    print(f"Split: {split_name}")
    print(f"Rows: {len(split_data):,}")
    print(f"Columns: {split_data.column_names}")
    
    if "db_id" in split_data.column_names:
        unique_databases = len(set(split_data["db_id"]))
        print(f"Unique databases: {unique_databases}")
    
    print("Missing values:")
    for column in split_data.column_names:
        missing_count = sum(value is None for value in split_data[column])
        print(f"  {column}: {missing_count}")

# Check whether database IDs overlap between splits
split_names = list(spider.keys())

if len(split_names) >= 2:
    first_split = split_names[0]
    second_split = split_names[1]

    first_databases = set(spider[first_split]["db_id"])
    second_databases = set(spider[second_split]["db_id"])
    database_overlap = first_databases.intersection(second_databases)

    print(f"\n{'=' * 60}")
    print(
        f"Database overlap between '{first_split}' "
        f"and '{second_split}': {len(database_overlap)}"
    )

    if database_overlap:
        print("WARNING: Database leakage detected.")
        print("Overlapping database IDs:", sorted(database_overlap)[:20])
    else:
        print("Database-level separation: PASSED")

# Display one example without printing an excessively long schema
first_split = split_names[0]
example = spider[first_split][0]

print(f"\n{'=' * 60}")
print("Example record")

for key, value in example.items():
    value_text = str(value)
    if len(value_text) > 700:
        value_text = value_text[:700] + "\n...[truncated]"
    print(f"\n{key}:\n{value_text}")

Dataset loaded successfully
DatasetDict({
    train: Dataset({
        features: ['query', 'Input', 'Question', 'Schema', 'db_id'],
        num_rows: 8659
    })
    validation: Dataset({
        features: ['query', 'Input', 'Question', 'Schema', 'db_id'],
        num_rows: 1034
    })
})

Split: train
Rows: 8,659
Columns: ['query', 'Input', 'Question', 'Schema', 'db_id']
Unique databases: 146
Missing values:
  query: 0
  Input: 0
  Question: 0
  Schema: 0
  db_id: 0

Split: validation
Rows: 1,034
Columns: ['query', 'Input', 'Question', 'Schema', 'db_id']
Unique databases: 20
Missing values:
  query: 0
  Input: 0
  Question: 0
  Schema: 0
  db_id: 0

Database overlap between 'train' and 'validation': 0
Database-level separation: PASSED

Example record

query:
SELECT count(*) FROM head WHERE age  >  56

Input:
Question: How many heads of the departments are older than 56 ?
Schema:
CREATE TABLE IF NOT EXISTS "department" (
"Department_ID" int,
"Name" text,
"Creation" text,
"Ranking" int,

## 4. Clean and Standardize the Dataset

The raw question and schema fields contain repeated labels. This section cleans the fields, removes exact duplicate examples, and converts every example into a consistent conversational Text-to-SQL format.

In [5]:
from datasets import Dataset, DatasetDict

SYSTEM_PROMPT = (
    "You are an expert SQLite developer. Given a database schema and a "
    "natural-language question, generate one valid SQL query that answers "
    "the question. Return only the SQL query without explanations, comments, "
    "or Markdown formatting."
)


def remove_leading_label(text, label):
    text = str(text).strip()
    prefix = f"{label}:"

    if text.lower().startswith(prefix.lower()):
        text = text[len(prefix):].strip()

    return text


def clean_split(split_data):
    cleaned_rows = []

    for row in split_data:
        question = remove_leading_label(row["Question"], "Question")
        schema = remove_leading_label(row["Schema"], "Schema")
        sql = str(row["query"]).strip().rstrip(";")

        cleaned_rows.append(
            {
                "db_id": str(row["db_id"]).strip(),
                "question": question,
                "schema": schema,
                "sql": sql,
            }
        )

    cleaned_dataset = Dataset.from_list(cleaned_rows)

    # Remove only completely identical examples
    dataframe = cleaned_dataset.to_pandas()
    rows_before = len(dataframe)

    dataframe = dataframe.drop_duplicates(
        subset=["db_id", "question", "schema", "sql"]
    ).reset_index(drop=True)

    rows_after = len(dataframe)

    print(f"Rows before deduplication: {rows_before:,}")
    print(f"Exact duplicates removed: {rows_before - rows_after:,}")
    print(f"Rows after deduplication: {rows_after:,}")

    return Dataset.from_pandas(dataframe, preserve_index=False)


cleaned_spider = DatasetDict(
    {
        split_name: clean_split(split_data)
        for split_name, split_data in spider.items()
    }
)


def add_messages(example):
    user_prompt = (
        f"Database schema:\n{example['schema']}\n\n"
        f"Question:\n{example['question']}\n\n"
        "Return only the SQL query."
    )

    return {
        "messages": [
            {
                "role": "system",
                "content": SYSTEM_PROMPT,
            },
            {
                "role": "user",
                "content": user_prompt,
            },
            {
                "role": "assistant",
                "content": example["sql"],
            },
        ]
    }


prepared_spider = cleaned_spider.map(add_messages)

print("\nPrepared dataset:")
print(prepared_spider)

# Confirm database separation again after cleaning
train_databases = set(prepared_spider["train"]["db_id"])
validation_databases = set(prepared_spider["validation"]["db_id"])

assert train_databases.isdisjoint(validation_databases)
print("\nDatabase-level separation after cleaning: PASSED")

# Confirm that required fields are non-empty
for split_name, split_data in prepared_spider.items():
    assert all(value.strip() for value in split_data["question"])
    assert all(value.strip() for value in split_data["schema"])
    assert all(value.strip() for value in split_data["sql"])
    print(f"{split_name} non-empty field validation: PASSED")

# Show one formatted conversation
example = prepared_spider["train"][0]

print("\nExample database ID:")
print(example["db_id"])

print("\nFormatted conversation:")
for message in example["messages"]:
    print(f"\n[{message['role'].upper()}]")
    print(message["content"])

Rows before deduplication: 8,659
Exact duplicates removed: 8
Rows after deduplication: 8,651
Rows before deduplication: 1,034
Exact duplicates removed: 0
Rows after deduplication: 1,034


Map:   0%|          | 0/8651 [00:00<?, ? examples/s]

Map:   0%|          | 0/1034 [00:00<?, ? examples/s]


Prepared dataset:
DatasetDict({
    train: Dataset({
        features: ['db_id', 'question', 'schema', 'sql', 'messages'],
        num_rows: 8651
    })
    validation: Dataset({
        features: ['db_id', 'question', 'schema', 'sql', 'messages'],
        num_rows: 1034
    })
})

Database-level separation after cleaning: PASSED
train non-empty field validation: PASSED
validation non-empty field validation: PASSED

Example database ID:
department_management

Formatted conversation:

[SYSTEM]
You are an expert SQLite developer. Given a database schema and a natural-language question, generate one valid SQL query that answers the question. Return only the SQL query without explanations, comments, or Markdown formatting.

[USER]
Database schema:
CREATE TABLE IF NOT EXISTS "department" (
"Department_ID" int,
"Name" text,
"Creation" text,
"Ranking" int,
"Budget_in_Billions" real,
"Num_Employees" real,
PRIMARY KEY ("Department_ID")
);
CREATE TABLE IF NOT EXISTS "head" (
"head_ID" int,
"nam

## 5. Load Qwen3-4B in 4-Bit Precision

Qwen3-4B is loaded using Unsloth's dynamic 4-bit quantization. The base model remains unchanged at this stage so it can later serve as the evaluation baseline.

In [6]:
import torch
from unsloth import FastLanguageModel

MODEL_NAME = "unsloth/Qwen3-4B-unsloth-bnb-4bit"
MAX_SEQ_LENGTH = 2048

torch.cuda.empty_cache()
torch.cuda.reset_peak_memory_stats()

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name=MODEL_NAME,
    max_seq_length=MAX_SEQ_LENGTH,
    dtype=None,          # Automatically uses FP16 on Tesla T4
    load_in_4bit=True,
)

if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

tokenizer.padding_side = "right"

total_parameters = sum(parameter.numel() for parameter in model.parameters())
trainable_parameters = sum(
    parameter.numel()
    for parameter in model.parameters()
    if parameter.requires_grad
)

allocated_memory = torch.cuda.memory_allocated(0) / (1024**3)
reserved_memory = torch.cuda.memory_reserved(0) / (1024**3)
peak_memory = torch.cuda.max_memory_allocated(0) / (1024**3)

print("\nModel loading verification")
print("-" * 50)
print("Model:", MODEL_NAME)
print("Maximum sequence length:", MAX_SEQ_LENGTH)
print("Model device:", next(model.parameters()).device)
print("Tokenizer vocabulary size:", len(tokenizer))
print("Padding token:", repr(tokenizer.pad_token))
print(f"Total parameters: {total_parameters:,}")
print(f"Trainable parameters before LoRA: {trainable_parameters:,}")
print(f"GPU memory allocated: {allocated_memory:.2f} GB")
print(f"GPU memory reserved: {reserved_memory:.2f} GB")
print(f"Peak GPU memory allocated: {peak_memory:.2f} GB")

assert model.is_loaded_in_4bit
assert next(model.parameters()).is_cuda

print("4-bit model verification: PASSED")

==((====))==  Unsloth 2026.9.4: Fast Qwen3 patching. Transformers: 5.5.0.
   \\   /|    Tesla T4. Num GPUs = 1. Max memory: 14.562 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.10.0+cu128. CUDA: 7.5. CUDA Toolkit: 12.8. Triton: 3.6.0
\        /    Bfloat16 = FALSE. FA [Xformers = 0.0.35. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


Loading weights:   0%|          | 0/398 [00:00<?, ?it/s]


Model loading verification
--------------------------------------------------
Model: unsloth/Qwen3-4B-unsloth-bnb-4bit
Maximum sequence length: 2048
Model device: cuda:0
Tokenizer vocabulary size: 151669
Padding token: '<|vision_pad|>'
Total parameters: 2,508,586,496
Trainable parameters before LoRA: 605,748,736
GPU memory allocated: 3.35 GB
GPU memory reserved: 3.39 GB
Peak GPU memory allocated: 3.37 GB
4-bit model verification: PASSED


## 6. Configure QLoRA Adapters

The quantized base model is explicitly frozen. Rank-16 LoRA adapters are attached to the attention and MLP projection layers, ensuring that only parameter-efficient adapter weights are updated during training.

In [7]:
from unsloth import FastLanguageModel

# Correct the padding configuration for text-only causal-LM training
original_pad_token = tokenizer.pad_token
tokenizer.pad_token = tokenizer.eos_token
tokenizer.padding_side = "right"

model.config.pad_token_id = tokenizer.pad_token_id

# Explicitly freeze every existing base-model parameter
for parameter in model.parameters():
    parameter.requires_grad = False

base_trainable_parameters = sum(
    parameter.numel()
    for parameter in model.parameters()
    if parameter.requires_grad
)

assert base_trainable_parameters == 0

# Attach LoRA adapters
model = FastLanguageModel.get_peft_model(
    model,
    r=16,
    target_modules=[
        "q_proj",
        "k_proj",
        "v_proj",
        "o_proj",
        "gate_proj",
        "up_proj",
        "down_proj",
    ],
    lora_alpha=32,
    lora_dropout=0,
    bias="none",
    use_gradient_checkpointing="unsloth",
    random_state=3407,
    use_rslora=False,
    loftq_config=None,
)

trainable_parameters = sum(
    parameter.numel()
    for parameter in model.parameters()
    if parameter.requires_grad
)

# Confirm that every trainable parameter belongs to a LoRA adapter
unexpected_trainable_parameters = [
    name
    for name, parameter in model.named_parameters()
    if parameter.requires_grad and "lora_" not in name.lower()
]

print("QLoRA configuration")
print("-" * 50)
print("Original padding token:", repr(original_pad_token))
print("New padding token:", repr(tokenizer.pad_token))
print("Padding token ID:", tokenizer.pad_token_id)
print("LoRA rank: 16")
print("LoRA alpha: 32")
print(f"Trainable adapter parameters: {trainable_parameters:,}")
print(
    "Approximate trainable percentage of the 4B architecture: "
    f"{100 * trainable_parameters / 4_000_000_000:.3f}%"
)
print(
    "Unexpected non-LoRA trainable parameters:",
    len(unexpected_trainable_parameters),
)

assert trainable_parameters > 0
assert len(unexpected_trainable_parameters) == 0

print("Base-model freezing: PASSED")
print("LoRA-only training verification: PASSED")

Unsloth 2026.9.4 patched 36 layers with 36 QKV layers, 36 O layers and 36 MLP layers.


QLoRA configuration
--------------------------------------------------
Original padding token: '<|vision_pad|>'
New padding token: '<|im_end|>'
Padding token ID: 151645
LoRA rank: 16
LoRA alpha: 32
Trainable adapter parameters: 33,030,144
Approximate trainable percentage of the 4B architecture: 0.826%
Unexpected non-LoRA trainable parameters: 0
Base-model freezing: PASSED
LoRA-only training verification: PASSED


## 7. Apply the Qwen3 Chat Template and Audit Sequence Lengths

Each conversation is rendered using Qwen3's native chat template with thinking mode disabled. Token lengths are measured without truncation to determine whether the selected context limit preserves the dataset adequately.

In [8]:
import numpy as np


def apply_qwen_chat_template(example):
    formatted_text = tokenizer.apply_chat_template(
        example["messages"],
        tokenize=False,
        add_generation_prompt=False,
        enable_thinking=False,
    )

    return {"text": formatted_text}


formatted_spider = prepared_spider.map(
    apply_qwen_chat_template,
    desc="Applying Qwen3 chat template",
)


def calculate_token_lengths(batch):
    tokenized = tokenizer(
        batch["text"],
        add_special_tokens=False,
        truncation=False,
    )

    return {
        "token_length": [
            len(input_ids)
            for input_ids in tokenized["input_ids"]
        ]
    }


formatted_spider = formatted_spider.map(
    calculate_token_lengths,
    batched=True,
    batch_size=128,
    desc="Calculating token lengths",
)

print("Token-length audit")
print("=" * 60)

for split_name, split_data in formatted_spider.items():
    lengths = np.array(split_data["token_length"])

    over_limit = int((lengths > MAX_SEQ_LENGTH).sum())
    over_limit_percentage = 100 * over_limit / len(lengths)

    print(f"\nSplit: {split_name}")
    print(f"Examples: {len(lengths):,}")
    print(f"Minimum length: {lengths.min():,}")
    print(f"Median length: {np.median(lengths):,.0f}")
    print(f"95th percentile: {np.percentile(lengths, 95):,.0f}")
    print(f"99th percentile: {np.percentile(lengths, 99):,.0f}")
    print(f"Maximum length: {lengths.max():,}")
    print(
        f"Examples over {MAX_SEQ_LENGTH:,} tokens: "
        f"{over_limit:,} ({over_limit_percentage:.2f}%)"
    )

# Display the beginning and ending of one rendered example
sample_text = formatted_spider["train"][0]["text"]

print("\n" + "=" * 60)
print("Rendered Qwen3 training example")
print("=" * 60)
print(sample_text[:1500])

if len(sample_text) > 1500:
    print("\n...[middle truncated for display]...\n")
    print(sample_text[-500:])

# Verify that essential chat-template markers exist
assert "<|im_start|>system" in sample_text
assert "<|im_start|>user" in sample_text
assert "<|im_start|>assistant" in sample_text
assert formatted_spider["train"][0]["sql"] in sample_text

print("\nChat-template verification: PASSED")

Applying Qwen3 chat template:   0%|          | 0/8651 [00:00<?, ? examples/s]

Applying Qwen3 chat template:   0%|          | 0/1034 [00:00<?, ? examples/s]

Calculating token lengths:   0%|          | 0/8651 [00:00<?, ? examples/s]

Calculating token lengths:   0%|          | 0/1034 [00:00<?, ? examples/s]

Token-length audit

Split: train
Examples: 8,651
Minimum length: 161
Median length: 490
95th percentile: 1,308
99th percentile: 2,206
Maximum length: 2,695
Examples over 2,048 tokens: 96 (1.11%)

Split: validation
Examples: 1,034
Minimum length: 190
Median length: 349
95th percentile: 934
99th percentile: 1,002
Maximum length: 1,024
Examples over 2,048 tokens: 0 (0.00%)

Rendered Qwen3 training example
<|im_start|>system
You are an expert SQLite developer. Given a database schema and a natural-language question, generate one valid SQL query that answers the question. Return only the SQL query without explanations, comments, or Markdown formatting.<|im_end|>
<|im_start|>user
Database schema:
CREATE TABLE IF NOT EXISTS "department" (
"Department_ID" int,
"Name" text,
"Creation" text,
"Ranking" int,
"Budget_in_Billions" real,
"Num_Employees" real,
PRIMARY KEY ("Department_ID")
);
CREATE TABLE IF NOT EXISTS "head" (
"head_ID" int,
"name" text,
"born_state" text,
"age" real,
PRIMARY KEY ("h

## 8. Finalize the Training Set and Test Base-Model Inference

Training examples exceeding the 2,048-token context limit are removed to prevent schema or target-SQL truncation. The untrained adapter is then used to reproduce the base model for an inference smoke test.

In [9]:
import re
import torch
from unsloth import FastLanguageModel

# Retain only complete examples that fit the configured context window
train_dataset = formatted_spider["train"].filter(
    lambda example: example["token_length"] <= MAX_SEQ_LENGTH,
    desc="Removing overlength training examples",
)

validation_dataset = formatted_spider["validation"]

print("Final training examples:", f"{len(train_dataset):,}")
print("Removed training examples:", f"{len(formatted_spider['train']) - len(train_dataset):,}")
print("Validation examples:", f"{len(validation_dataset):,}")

assert len(train_dataset) == 8555
assert max(train_dataset["token_length"]) <= MAX_SEQ_LENGTH
assert max(validation_dataset["token_length"]) <= MAX_SEQ_LENGTH

print("Sequence-length filtering: PASSED")

# Confirm that LoRA B matrices are still zero-initialized.
# Therefore, the current model behaves like the untouched base model.
lora_b_parameters = [
    parameter
    for name, parameter in model.named_parameters()
    if "lora_B" in name
]

assert lora_b_parameters

maximum_lora_b_value = max(
    parameter.detach().abs().max().item()
    for parameter in lora_b_parameters
)

print(f"Maximum initial LoRA-B value: {maximum_lora_b_value:.8f}")
assert maximum_lora_b_value == 0.0
print("Untouched base-model equivalence: PASSED")


def build_inference_prompt(example):
    # Exclude the gold assistant response from the inference input
    inference_messages = example["messages"][:-1]

    return tokenizer.apply_chat_template(
        inference_messages,
        tokenize=False,
        add_generation_prompt=True,
        enable_thinking=False,
    )


def clean_sql_prediction(text):
    text = text.strip()

    # Defensive removal in case a thinking block appears in the decoded text
    text = re.sub(
        r"<think>.*?</think>",
        "",
        text,
        flags=re.DOTALL | re.IGNORECASE,
    ).strip()

    # Remove Markdown code fences if the base model ignores the instruction
    text = re.sub(r"^```(?:sql)?\s*", "", text, flags=re.IGNORECASE)
    text = re.sub(r"\s*```$", "", text).strip()

    return text.rstrip(";").strip()


# Select three examples from different validation databases
sample_indices = []
seen_databases = set()

for index, database_id in enumerate(validation_dataset["db_id"]):
    if database_id not in seen_databases:
        sample_indices.append(index)
        seen_databases.add(database_id)

    if len(sample_indices) == 3:
        break

FastLanguageModel.for_inference(model)
tokenizer.padding_side = "left"

model_device = next(model.parameters()).device

print("\nBase-model inference smoke test")
print("=" * 70)

for sample_number, index in enumerate(sample_indices, start=1):
    example = validation_dataset[index]
    prompt = build_inference_prompt(example)

    inputs = tokenizer(
        prompt,
        return_tensors="pt",
        add_special_tokens=False,
        truncation=False,
    ).to(model_device)

    prompt_length = inputs["input_ids"].shape[1]

    with torch.inference_mode():
        generated = model.generate(
            **inputs,
            max_new_tokens=256,
            do_sample=False,
            use_cache=True,
            pad_token_id=tokenizer.pad_token_id,
            eos_token_id=tokenizer.eos_token_id,
        )

    generated_tokens = generated[0, prompt_length:]
    raw_prediction = tokenizer.decode(
        generated_tokens,
        skip_special_tokens=True,
    )

    prediction = clean_sql_prediction(raw_prediction)

    print(f"\nExample {sample_number}")
    print("-" * 70)
    print("Database:", example["db_id"])
    print("Question:", example["question"])
    print("Gold SQL:", example["sql"])
    print("Base prediction:", prediction)

print("\nBase-model inference smoke test: COMPLETED")

Removing overlength training examples:   0%|          | 0/8651 [00:00<?, ? examples/s]

Final training examples: 8,555
Removed training examples: 96
Validation examples: 1,034
Sequence-length filtering: PASSED
Maximum initial LoRA-B value: 0.00000000
Untouched base-model equivalence: PASSED

Base-model inference smoke test


Both `max_new_tokens` (=256) and `max_length`(=40960) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `max_new_tokens` (=256) and `max_length`(=40960) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)



Example 1
----------------------------------------------------------------------
Database: concert_singer
Question: How many singers do we have?
Gold SQL: SELECT count(*) FROM singer
Base prediction: SELECT COUNT(*) FROM singer


Both `max_new_tokens` (=256) and `max_length`(=40960) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)



Example 2
----------------------------------------------------------------------
Database: pets_1
Question: Find the number of pets whose weight is heavier than 10.
Gold SQL: SELECT count(*) FROM pets WHERE weight  >  10
Base prediction: SELECT COUNT(*) FROM Has_Pet hp JOIN Pets p ON hp.PetID = p.PetID WHERE p.weight > 10

Example 3
----------------------------------------------------------------------
Database: car_1
Question: How many continents are there?
Gold SQL: SELECT count(*) FROM CONTINENTS
Base prediction: SELECT COUNT(*) FROM continents

Base-model inference smoke test: COMPLETED


In [10]:
%pip install -q sqlglot

Note: you may need to restart the kernel to use updated packages.


In [11]:
model_device = next(model.parameters()).device
torch.cuda.set_device(model_device)

print("Model device:", model_device)
print("Active CUDA device:", torch.cuda.current_device())

assert torch.cuda.current_device() == model_device.index
print("CUDA device alignment: PASSED")

Model device: cuda:0
Active CUDA device: 0
CUDA device alignment: PASSED


## 9. Evaluate the Untouched Base Model

The base model is evaluated on all 1,034 unseen-schema validation examples. Metrics include SQL syntax validity, output-format compliance, and SQLGlot-normalized exact-match accuracy. Predictions are saved for comparison with the fine-tuned model.

In [12]:
import re
import time
import sqlglot
import pandas as pd
import torch

from tqdm.auto import tqdm
from unsloth import FastLanguageModel


def clean_sql_prediction(text):
    text = str(text).strip()

    # Remove optional Qwen thinking content
    text = re.sub(
        r"<think>.*?</think>",
        "",
        text,
        flags=re.DOTALL | re.IGNORECASE,
    ).strip()

    # Remove Markdown code fences
    text = re.sub(
        r"^```(?:sql)?\s*",
        "",
        text,
        flags=re.IGNORECASE,
    )

    text = re.sub(
        r"\s*```$",
        "",
        text,
    ).strip()

    return text.rstrip(";").strip()


def canonicalize_sql(sql):
    sql = clean_sql_prediction(sql)

    if not sql:
        return None

    try:
        expression = sqlglot.parse_one(
            sql,
            read="sqlite",
        )

        if expression is None:
            return None

        return expression.sql(
            dialect="sqlite",
            pretty=False,
            normalize=True,
        )

    except Exception:
        return None


def follows_sql_only_format(sql):
    sql = clean_sql_prediction(sql)
    upper_sql = sql.upper()

    begins_with_sql = (
        upper_sql.startswith("SELECT ")
        or upper_sql.startswith("WITH ")
        or upper_sql == "SELECT"
    )

    contains_markdown = "```" in sql
    contains_multiple_statements = ";" in sql

    return (
        bool(sql)
        and begins_with_sql
        and not contains_markdown
        and not contains_multiple_statements
    )


def count_generated_tokens(token_ids, eos_token_id):
    token_ids = token_ids.tolist()

    if eos_token_id in token_ids:
        return token_ids.index(eos_token_id) + 1

    return len(token_ids)


# Enable Unsloth's optimized inference mode
FastLanguageModel.for_inference(model)
model.eval()
tokenizer.padding_side = "left"

BATCH_SIZE = 8
MAX_NEW_TOKENS = 256

# Keep every inference tensor on cuda:0
model_device = next(model.parameters()).device
torch.cuda.set_device(model_device)

print("Model device:", model_device)
print("Active CUDA device:", torch.cuda.current_device())

assert torch.cuda.device_count() == 1
assert model_device.type == "cuda"
assert model_device.index == 0
assert torch.cuda.current_device() == 0

print("CUDA device alignment: PASSED")

base_records = []

torch.cuda.empty_cache()
torch.cuda.reset_peak_memory_stats(model_device)

evaluation_start = time.time()

for start_index in tqdm(
    range(0, len(validation_dataset), BATCH_SIZE),
    desc="Evaluating untouched base model",
):
    end_index = min(
        start_index + BATCH_SIZE,
        len(validation_dataset),
    )

    batch = validation_dataset[start_index:end_index]

    prompts = [
        tokenizer.apply_chat_template(
            messages[:-1],
            tokenize=False,
            add_generation_prompt=True,
            enable_thinking=False,
        )
        for messages in batch["messages"]
    ]

    inputs = tokenizer(
        prompts,
        return_tensors="pt",
        padding=True,
        add_special_tokens=False,
        truncation=False,
    )

    inputs = {
        key: value.to(model_device)
        for key, value in inputs.items()
    }

    input_length = inputs["input_ids"].shape[1]

    with torch.inference_mode():
        generated = model.generate(
            **inputs,
            max_length=input_length + MAX_NEW_TOKENS,
            do_sample=False,
            num_beams=1,
            use_cache=True,
            pad_token_id=tokenizer.pad_token_id,
            eos_token_id=tokenizer.eos_token_id,
        )

    generated_batch = generated[:, input_length:]

    for batch_position in range(len(prompts)):
        generated_tokens = generated_batch[batch_position]

        raw_prediction = tokenizer.decode(
            generated_tokens,
            skip_special_tokens=True,
        )

        prediction = clean_sql_prediction(raw_prediction)
        gold_sql = batch["sql"][batch_position]

        canonical_prediction = canonicalize_sql(prediction)
        canonical_gold = canonicalize_sql(gold_sql)

        syntax_valid = canonical_prediction is not None
        format_compliant = follows_sql_only_format(prediction)

        normalized_exact_match = (
            syntax_valid
            and canonical_gold is not None
            and canonical_prediction == canonical_gold
        )

        base_records.append(
            {
                "db_id": batch["db_id"][batch_position],
                "question": batch["question"][batch_position],
                "gold_sql": gold_sql,
                "base_prediction": prediction,
                "canonical_gold": canonical_gold,
                "canonical_base_prediction": canonical_prediction,
                "syntax_valid": syntax_valid,
                "format_compliant": format_compliant,
                "normalized_exact_match": normalized_exact_match,
                "generated_tokens": count_generated_tokens(
                    generated_tokens,
                    tokenizer.eos_token_id,
                ),
            }
        )

evaluation_minutes = (
    time.time() - evaluation_start
) / 60

base_results = pd.DataFrame(base_records)

base_syntax_validity = (
    100 * base_results["syntax_valid"].mean()
)

base_format_compliance = (
    100 * base_results["format_compliant"].mean()
)

base_exact_match = (
    100 * base_results["normalized_exact_match"].mean()
)

base_average_tokens = (
    base_results["generated_tokens"].mean()
)

peak_gpu_memory = (
    torch.cuda.max_memory_allocated(model_device)
    / (1024**3)
)

BASE_RESULTS_PATH = (
    "/kaggle/working/base_model_predictions.csv"
)

base_results.to_csv(
    BASE_RESULTS_PATH,
    index=False,
)

print("\nUntouched base-model results")
print("=" * 60)
print(f"Validation examples: {len(base_results):,}")
print(f"SQL syntax validity: {base_syntax_validity:.2f}%")
print(f"SQL-only format compliance: {base_format_compliance:.2f}%")
print(f"Normalized exact match: {base_exact_match:.2f}%")
print(f"Average generated tokens: {base_average_tokens:.2f}")
print(f"Peak GPU memory: {peak_gpu_memory:.2f} GB")
print(f"Evaluation time: {evaluation_minutes:.2f} minutes")
print(f"Predictions saved to: {BASE_RESULTS_PATH}")

assert len(base_results) == len(validation_dataset)

print("\nBase-model evaluation completeness: PASSED")

incorrect_examples = base_results[
    ~base_results["normalized_exact_match"]
].head(5)

print("\nFirst five normalized-exact-match errors")
print("=" * 60)

for _, row in incorrect_examples.iterrows():
    print(f"\nDatabase: {row['db_id']}")
    print(f"Question: {row['question']}")
    print(f"Gold SQL: {row['gold_sql']}")
    print(f"Base prediction: {row['base_prediction']}")

Model device: cuda:0
Active CUDA device: 0
CUDA device alignment: PASSED


Evaluating untouched base model:   0%|          | 0/130 [00:00<?, ?it/s]


Untouched base-model results
Validation examples: 1,034
SQL syntax validity: 99.61%
SQL-only format compliance: 99.71%
Normalized exact match: 22.63%
Average generated tokens: 30.28
Peak GPU memory: 6.36 GB
Evaluation time: 28.86 minutes
Predictions saved to: /kaggle/working/base_model_predictions.csv

Base-model evaluation completeness: PASSED

First five normalized-exact-match errors

Database: concert_singer
Question: What is the average, minimum, and maximum age of all singers from France?
Gold SQL: SELECT avg(age) ,  min(age) ,  max(age) FROM singer WHERE country  =  'France'
Base prediction: SELECT AVG(Age) AS Average, MIN(Age) AS Minimum, MAX(Age) AS Maximum FROM singer WHERE Country = 'France'

Database: concert_singer
Question: What is the average, minimum, and maximum age for all French singers?
Gold SQL: SELECT avg(age) ,  min(age) ,  max(age) FROM singer WHERE country  =  'France'
Base prediction: SELECT AVG(Age) AS Average, MIN(Age) AS Minimum, MAX(Age) AS Maximum FROM si

In [13]:
import inspect
import trl

from trl import SFTConfig, SFTTrainer

print("TRL version:", trl.__version__)
print("\nSFTConfig parameters:")
print(inspect.signature(SFTConfig))
print("\nSFTTrainer import: PASSED")

TRL version: 0.24.0

SFTConfig parameters:
(output_dir=None, per_device_train_batch_size=4, num_train_epochs=3.0, max_steps=-1, learning_rate=5e-05, lr_scheduler_type='linear', lr_scheduler_kwargs=None, warmup_steps=0.1, optim='adamw_8bit', optim_args=None, weight_decay=0.001, adam_beta1=0.9, adam_beta2=0.999, adam_epsilon=1e-08, optim_target_modules=None, gradient_accumulation_steps=2, average_tokens_across_devices=True, max_grad_norm=1.0, label_smoothing_factor=0.0, bf16=False, fp16=False, bf16_full_eval=False, fp16_full_eval=False, tf32=None, gradient_checkpointing=True, gradient_checkpointing_kwargs=None, torch_compile=False, torch_compile_backend=None, torch_compile_mode=None, use_liger_kernel=False, liger_kernel_config=None, use_cache=False, neftune_noise_alpha=None, torch_empty_cache_steps=250, auto_find_batch_size=False, logging_strategy='steps', logging_steps=1, logging_first_step=False, log_on_each_node=True, logging_nan_inf_filter=False, include_num_input_tokens_seen=False, 

## 10. Fine-Tune with Unsloth-Accelerated QLoRA

The rank-16 adapters are trained for 100 optimizer steps using assistant-response-only loss, FP16 computation, 8-bit AdamW, and Unsloth gradient checkpointing.

In [14]:
import json
import time
import pandas as pd
import torch

from trl import SFTConfig, SFTTrainer
from unsloth import FastLanguageModel
from unsloth.chat_templates import train_on_responses_only

# Switch from inference mode to training mode
FastLanguageModel.for_training(model)
model.train()
tokenizer.padding_side = "right"

TRAINING_STEPS = 100
PER_DEVICE_BATCH_SIZE = 2
GRADIENT_ACCUMULATION_STEPS = 4
EFFECTIVE_BATCH_SIZE = (
    PER_DEVICE_BATCH_SIZE
    * GRADIENT_ACCUMULATION_STEPS
)

training_config = SFTConfig(
    output_dir="/kaggle/working/qwen3_text_to_sql_training",
    dataset_text_field="text",
    max_length=MAX_SEQ_LENGTH,
    max_seq_length=MAX_SEQ_LENGTH,
    dataset_num_proc=2,
    packing=False,

    per_device_train_batch_size=PER_DEVICE_BATCH_SIZE,
    gradient_accumulation_steps=GRADIENT_ACCUMULATION_STEPS,
    max_steps=TRAINING_STEPS,

    learning_rate=2e-4,
    warmup_steps=5,
    optim="adamw_8bit",
    weight_decay=0.01,
    lr_scheduler_type="linear",
    max_grad_norm=1.0,

    fp16=True,
    bf16=False,

    logging_steps=5,
    logging_first_step=True,
    save_strategy="no",
    eval_strategy="no",
    report_to="none",

    seed=3407,
    data_seed=3407,
)

trainer = SFTTrainer(
    model=model,
    processing_class=tokenizer,
    train_dataset=train_dataset,
    args=training_config,
)

# Train only on the target SQL, not the system prompt or schema
trainer = train_on_responses_only(
    trainer,
    instruction_part="<|im_start|>user\n",
    response_part=(
        "<|im_start|>assistant\n"
        "<think>\n\n</think>\n\n"
    ),
)

# Verify response-only masking before training
mask_example = trainer.train_dataset[0]

trained_token_ids = [
    token_id
    for token_id, label in zip(
        mask_example["input_ids"],
        mask_example["labels"],
    )
    if label != -100
]

trained_text = tokenizer.decode(
    trained_token_ids,
    skip_special_tokens=False,
)

print("Response-only training sample:")
print(trained_text)

assert trained_token_ids
assert "SELECT" in trained_text.upper()
assert "DATABASE SCHEMA" not in trained_text.upper()

print("\nResponse-only masking: PASSED")
print("Training steps:", TRAINING_STEPS)
print("Effective batch size:", EFFECTIVE_BATCH_SIZE)
print(
    "Approximate sequences processed:",
    TRAINING_STEPS * EFFECTIVE_BATCH_SIZE,
)

torch.cuda.empty_cache()
torch.cuda.reset_peak_memory_stats(model_device)

training_start = time.time()
training_result = trainer.train()
training_minutes = (time.time() - training_start) / 60

peak_training_memory = (
    torch.cuda.max_memory_allocated(model_device)
    / (1024**3)
)

training_history = pd.DataFrame(
    trainer.state.log_history
)

TRAINING_HISTORY_PATH = (
    "/kaggle/working/training_history.csv"
)

training_history.to_csv(
    TRAINING_HISTORY_PATH,
    index=False,
)

training_summary = {
    "model": MODEL_NAME,
    "framework": "Unsloth",
    "method": "4-bit QLoRA",
    "training_steps": TRAINING_STEPS,
    "effective_batch_size": EFFECTIVE_BATCH_SIZE,
    "trainable_parameters": 33030144,
    "training_minutes": round(training_minutes, 2),
    "peak_gpu_memory_gb": round(peak_training_memory, 2),
    "train_loss": training_result.metrics.get("train_loss"),
    "train_samples_per_second": training_result.metrics.get(
        "train_samples_per_second"
    ),
    "train_steps_per_second": training_result.metrics.get(
        "train_steps_per_second"
    ),
}

TRAINING_SUMMARY_PATH = (
    "/kaggle/working/training_summary.json"
)

with open(
    TRAINING_SUMMARY_PATH,
    "w",
    encoding="utf-8",
) as file:
    json.dump(
        training_summary,
        file,
        indent=2,
    )

print("\nUnsloth training completed")
print("=" * 60)

for key, value in training_summary.items():
    print(f"{key}: {value}")

print(
    "\nTraining history saved to:",
    TRAINING_HISTORY_PATH,
)
print(
    "Training summary saved to:",
    TRAINING_SUMMARY_PATH,
)

Unsloth: Tokenizing ["text"] (num_proc=2):   0%|          | 0/8555 [00:00<?, ? examples/s]

🦥 Unsloth: Padding-free auto-enabled, enabling faster training.


Map (num_proc=2):   0%|          | 0/8555 [00:00<?, ? examples/s]

The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'bos_token_id': None, 'pad_token_id': 151645}.


Response-only training sample:
SELECT count(*) FROM head WHERE age  >  56<|im_end|>


Response-only masking: PASSED
Training steps: 100
Effective batch size: 8
Approximate sequences processed: 800


==((====))==  Unsloth - 2x faster free finetuning | Num GPUs used = 1
   \\   /|    Num examples = 8,555 | Num Epochs = 1 | Total steps = 100
O^O/ \_/ \    Batch size per device = 2 | Gradient accumulation steps = 4
\        /    Data Parallel GPUs = 1 | Total batch size (2 x 4 x 1) = 8
 "-____-"     Trainable parameters = 33,030,144 of 4,055,498,240 (0.81% trained)
`use_return_dict` is deprecated! Use `return_dict` instead!


Unsloth: Will smartly offload gradients to save VRAM!
Unsloth: Double buffering enabled (parallel H2D + compute) for backward pass.


Step,Training Loss
1,3.832408
5,2.587168
10,0.876506
15,0.355580
20,0.327056
25,0.276349
30,0.220416
35,0.213967
40,0.233211
45,0.266848



Unsloth training completed
model: unsloth/Qwen3-4B-unsloth-bnb-4bit
framework: Unsloth
method: 4-bit QLoRA
training_steps: 100
effective_batch_size: 8
trainable_parameters: 33030144
training_minutes: 18.15
peak_gpu_memory_gb: 4.77
train_loss: 0.40030657112598417
train_samples_per_second: 0.736
train_steps_per_second: 0.092

Training history saved to: /kaggle/working/training_history.csv
Training summary saved to: /kaggle/working/training_summary.json


# STEP 11: Evaluate the fine-tuned Unsloth adapter

In [16]:
# ============================================================
# STEP 11: Evaluate the fine-tuned Unsloth adapter
# ============================================================

import gc
import json
import math
import re
import time

import numpy as np
import pandas as pd
import torch
import sqlglot

from tqdm.auto import tqdm
from unsloth import FastLanguageModel


# ------------------------------------------------------------
# 1. Evaluation configuration
# ------------------------------------------------------------

BATCH_SIZE = 8
MAX_NEW_TOKENS = 256
MAX_SEQ_LENGTH = 2048

PREDICTIONS_PATH = "/kaggle/working/finetuned_model_predictions.csv"
SUMMARY_PATH = "/kaggle/working/finetuned_evaluation_summary.json"

# Results from Step 9
BASELINE_RESULTS = {
    "syntax_validity": 99.61,
    "format_compliance": 99.71,
    "normalized_exact_match": 22.63,
    "peak_gpu_memory_gb": 6.36,
    "evaluation_minutes": 28.86,
}


# ------------------------------------------------------------
# 2. Select validation dataset
# ------------------------------------------------------------

validation_data = validation_dataset

assert len(validation_data) == 1034, (
    f"Expected 1,034 validation examples, "
    f"but found {len(validation_data):,}"
)

print("Validation examples:", len(validation_data))


# ------------------------------------------------------------
# 3. Configure the trained model for inference
# ------------------------------------------------------------

FastLanguageModel.for_inference(model)
model.eval()

tokenizer.padding_side = "left"

if tokenizer.pad_token_id is None:
    tokenizer.pad_token = tokenizer.eos_token

model.config.pad_token_id = tokenizer.pad_token_id

if hasattr(model, "generation_config"):
    model.generation_config.pad_token_id = tokenizer.pad_token_id
    model.generation_config.eos_token_id = tokenizer.eos_token_id

gc.collect()
torch.cuda.empty_cache()
torch.cuda.reset_peak_memory_stats()

model_device = next(model.parameters()).device
active_device = torch.cuda.current_device()

print("\nModel device:", model_device)
print("Active CUDA device:", active_device)

assert model_device.type == "cuda", "Model is not loaded on a CUDA device."
assert model_device.index in (None, active_device), (
    f"Device mismatch: model={model_device}, "
    f"active CUDA device=cuda:{active_device}"
)

print("CUDA device alignment: PASSED")


# ------------------------------------------------------------
# 4. SQL utility functions
# ------------------------------------------------------------

SQL_START_PATTERN = re.compile(
    r"\b(SELECT|WITH|INSERT|UPDATE|DELETE|CREATE|DROP|ALTER)\b",
    flags=re.IGNORECASE,
)


def clean_generated_sql(text):
    """
    Extract the SQL portion for syntax and exact-match evaluation.
    Format compliance is evaluated separately using the raw output.
    """

    text = str(text).strip()

    # Remove a generated thinking block if one appears
    if "</think>" in text:
        text = text.split("</think>", 1)[1].strip()

    # Remove Markdown fences for SQL-content evaluation
    text = re.sub(
        r"^\s*```(?:sql|sqlite)?\s*",
        "",
        text,
        flags=re.IGNORECASE,
    )
    text = re.sub(r"\s*```\s*$", "", text)

    # Remove common explanatory prefixes
    text = re.sub(
        r"^\s*(SQL query|SQL|Query|Answer)\s*:\s*",
        "",
        text,
        flags=re.IGNORECASE,
    )

    # Start from the first SQL keyword if extra text appears before it
    match = SQL_START_PATTERN.search(text)

    if match:
        text = text[match.start():]

    return text.strip()


def is_sql_only_format(raw_text):
    """
    Check whether the model returned SQL without Markdown or explanation.
    """

    text = str(raw_text).strip()

    if not text:
        return False

    if "```" in text:
        return False

    if "<think>" in text or "</think>" in text:
        return False

    explanatory_patterns = [
        r"^\s*here\s+is",
        r"^\s*the\s+(sql\s+)?query",
        r"^\s*sql\s*:",
        r"^\s*query\s*:",
        r"^\s*answer\s*:",
        r"^\s*explanation\s*:",
    ]

    for pattern in explanatory_patterns:
        if re.search(pattern, text, flags=re.IGNORECASE):
            return False

    return bool(re.match(
        r"^\s*(SELECT|WITH|INSERT|UPDATE|DELETE|CREATE|DROP|ALTER)\b",
        text,
        flags=re.IGNORECASE,
    ))


def is_valid_sql(sql_text):
    """
    Determine whether SQLGlot can parse the prediction as SQLite SQL.
    """

    if not sql_text:
        return False

    try:
        parsed_statements = sqlglot.parse(sql_text, read="sqlite")

        return (
            len(parsed_statements) == 1
            and parsed_statements[0] is not None
        )

    except Exception:
        return False


def normalize_sql(sql_text):
    """
    Normalize SQL through SQLGlot for strict normalized exact match.
    """

    if not sql_text:
        return None

    try:
        parsed = sqlglot.parse_one(sql_text, read="sqlite")

        normalized = parsed.sql(
            dialect="sqlite",
            pretty=False,
            normalize=True,
        )

        normalized = re.sub(r"\s+", " ", normalized)
        normalized = normalized.strip().rstrip(";").strip()

        return normalized

    except Exception:
        return None


# ------------------------------------------------------------
# 5. Build inference prompts
# ------------------------------------------------------------

def build_inference_prompt(example):
    messages = example["messages"]

    # Remove the gold assistant answer
    prompt_messages = [
        message
        for message in messages
        if message["role"] != "assistant"
    ]

    return tokenizer.apply_chat_template(
        prompt_messages,
        tokenize=False,
        add_generation_prompt=True,
        enable_thinking=False,
    )


prompts = [
    build_inference_prompt(example)
    for example in validation_data
]

print("\nInference prompts created:", len(prompts))


# ------------------------------------------------------------
# 6. Run batched generation
# ------------------------------------------------------------

predictions = []

total_batches = math.ceil(len(validation_data) / BATCH_SIZE)

torch.cuda.synchronize()
evaluation_start = time.time()

for start_index in tqdm(
    range(0, len(validation_data), BATCH_SIZE),
    total=total_batches,
    desc="Evaluating fine-tuned model",
):

    end_index = min(
        start_index + BATCH_SIZE,
        len(validation_data),
    )

    batch_prompts = prompts[start_index:end_index]

    inputs = tokenizer(
        batch_prompts,
        return_tensors="pt",
        padding=True,
        truncation=True,
        max_length=MAX_SEQ_LENGTH,
        add_special_tokens=False,
    )

    inputs = {
        key: value.to(model_device)
        for key, value in inputs.items()
    }

    input_length = inputs["input_ids"].shape[1]

    with torch.inference_mode():
        generated = model.generate(
            **inputs,
            max_new_tokens=MAX_NEW_TOKENS,
            do_sample=False,
            use_cache=True,
            eos_token_id=tokenizer.eos_token_id,
            pad_token_id=tokenizer.pad_token_id,
        )

    generated_tokens = generated[:, input_length:]

    batch_predictions = tokenizer.batch_decode(
        generated_tokens,
        skip_special_tokens=True,
        clean_up_tokenization_spaces=False,
    )

    predictions.extend(
        prediction.strip()
        for prediction in batch_predictions
    )


torch.cuda.synchronize()
evaluation_seconds = time.time() - evaluation_start
evaluation_minutes = evaluation_seconds / 60

assert len(predictions) == len(validation_data), (
    "The number of predictions does not match the validation dataset."
)


# ------------------------------------------------------------
# 7. Calculate evaluation metrics
# ------------------------------------------------------------

evaluation_records = []

for example, raw_prediction in zip(
    validation_data,
    predictions,
):

    gold_sql = str(example["sql"]).strip()
    predicted_sql = clean_generated_sql(raw_prediction)

    gold_normalized = normalize_sql(gold_sql)
    prediction_normalized = normalize_sql(predicted_sql)

    syntax_valid = is_valid_sql(predicted_sql)
    format_compliant = is_sql_only_format(raw_prediction)

    exact_match = (
        gold_normalized is not None
        and prediction_normalized is not None
        and gold_normalized == prediction_normalized
    )

    generated_token_count = len(
        tokenizer.encode(
            raw_prediction,
            add_special_tokens=False,
        )
    )

    evaluation_records.append({
        "db_id": example["db_id"],
        "question": example["question"],
        "gold_sql": gold_sql,
        "finetuned_prediction": predicted_sql,
        "raw_prediction": raw_prediction,
        "gold_normalized": gold_normalized,
        "prediction_normalized": prediction_normalized,
        "syntax_valid": syntax_valid,
        "format_compliant": format_compliant,
        "normalized_exact_match": exact_match,
        "generated_tokens": generated_token_count,
    })


results_df = pd.DataFrame(evaluation_records)

syntax_validity = (
    results_df["syntax_valid"].mean() * 100
)

format_compliance = (
    results_df["format_compliant"].mean() * 100
)

normalized_exact_match = (
    results_df["normalized_exact_match"].mean() * 100
)

average_generated_tokens = (
    results_df["generated_tokens"].mean()
)

peak_gpu_memory_gb = (
    torch.cuda.max_memory_allocated() / (1024 ** 3)
)


# ------------------------------------------------------------
# 8. Save predictions and summary
# ------------------------------------------------------------

results_df.to_csv(
    PREDICTIONS_PATH,
    index=False,
)

evaluation_summary = {
    "model": "unsloth/Qwen3-4B-unsloth-bnb-4bit",
    "framework": "Unsloth",
    "method": "4-bit QLoRA",
    "training_steps": 100,
    "validation_examples": len(results_df),
    "sql_syntax_validity_percent": round(
        syntax_validity, 4
    ),
    "sql_only_format_compliance_percent": round(
        format_compliance, 4
    ),
    "normalized_exact_match_percent": round(
        normalized_exact_match, 4
    ),
    "average_generated_tokens": round(
        average_generated_tokens, 4
    ),
    "peak_gpu_memory_gb": round(
        peak_gpu_memory_gb, 4
    ),
    "evaluation_minutes": round(
        evaluation_minutes, 4
    ),
    "base_normalized_exact_match_percent": 22.63,
    "exact_match_change_percentage_points": round(
        normalized_exact_match
        - BASELINE_RESULTS["normalized_exact_match"],
        4,
    ),
}

with open(SUMMARY_PATH, "w", encoding="utf-8") as file:
    json.dump(
        evaluation_summary,
        file,
        indent=2,
    )


# ------------------------------------------------------------
# 9. Display final results
# ------------------------------------------------------------

print("\nFine-tuned model results")
print("=" * 60)

print(
    f"Validation examples: "
    f"{len(results_df):,}"
)

print(
    f"SQL syntax validity: "
    f"{syntax_validity:.2f}%"
)

print(
    f"SQL-only format compliance: "
    f"{format_compliance:.2f}%"
)

print(
    f"Normalized exact match: "
    f"{normalized_exact_match:.2f}%"
)

print(
    f"Average generated tokens: "
    f"{average_generated_tokens:.2f}"
)

print(
    f"Peak GPU memory: "
    f"{peak_gpu_memory_gb:.2f} GB"
)

print(
    f"Evaluation time: "
    f"{evaluation_minutes:.2f} minutes"
)

print(
    f"Predictions saved to: "
    f"{PREDICTIONS_PATH}"
)

print(
    f"Summary saved to: "
    f"{SUMMARY_PATH}"
)


# ------------------------------------------------------------
# 10. Compare against the untouched base model
# ------------------------------------------------------------

exact_match_change = (
    normalized_exact_match
    - BASELINE_RESULTS["normalized_exact_match"]
)

syntax_change = (
    syntax_validity
    - BASELINE_RESULTS["syntax_validity"]
)

format_change = (
    format_compliance
    - BASELINE_RESULTS["format_compliance"]
)

print("\nBase model vs fine-tuned model")
print("=" * 60)

comparison_df = pd.DataFrame({
    "Metric": [
        "SQL syntax validity (%)",
        "SQL-only format compliance (%)",
        "Normalized exact match (%)",
    ],
    "Base model": [
        BASELINE_RESULTS["syntax_validity"],
        BASELINE_RESULTS["format_compliance"],
        BASELINE_RESULTS["normalized_exact_match"],
    ],
    "Fine-tuned model": [
        round(syntax_validity, 2),
        round(format_compliance, 2),
        round(normalized_exact_match, 2),
    ],
    "Change (percentage points)": [
        round(syntax_change, 2),
        round(format_change, 2),
        round(exact_match_change, 2),
    ],
})

display(comparison_df)


# ------------------------------------------------------------
# 11. Show first five remaining exact-match errors
# ------------------------------------------------------------

errors_df = results_df[
    ~results_df["normalized_exact_match"]
].head(5)

print("\nFirst five normalized-exact-match errors")
print("=" * 60)

for _, row in errors_df.iterrows():

    print(f"\nDatabase: {row['db_id']}")
    print(f"Question: {row['question']}")
    print(f"Gold SQL: {row['gold_sql']}")
    print(
        f"Fine-tuned prediction: "
        f"{row['finetuned_prediction']}"
    )


# ------------------------------------------------------------
# 12. Final verification
# ------------------------------------------------------------

assert len(results_df) == 1034
assert results_df["finetuned_prediction"].notna().all()

print("\nFine-tuned evaluation completeness: PASSED")

Validation examples: 1034

Model device: cuda:0
Active CUDA device: 0
CUDA device alignment: PASSED

Inference prompts created: 1034


Evaluating fine-tuned model:   0%|          | 0/130 [00:00<?, ?it/s]

Both `max_new_tokens` (=256) and `max_length`(=40960) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `max_new_tokens` (=256) and `max_length`(=40960) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `max_new_tokens` (=256) and `max_length`(=40960) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `max_new_tokens` (=256) and `max_length`(=40960) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_gene


Fine-tuned model results
Validation examples: 1,034
SQL syntax validity: 100.00%
SQL-only format compliance: 100.00%
Normalized exact match: 39.46%
Average generated tokens: 29.99
Peak GPU memory: 6.46 GB
Evaluation time: 29.31 minutes
Predictions saved to: /kaggle/working/finetuned_model_predictions.csv
Summary saved to: /kaggle/working/finetuned_evaluation_summary.json

Base model vs fine-tuned model


,Metric,Base model,Fine-tuned model,Change (percentage points)
0,SQL syntax validity (%),99.61,100.00,0.39
1,SQL-only format compliance (%),99.71,100.00,0.29
2,Normalized exact match (%),22.63,39.46,16.83



First five normalized-exact-match errors

Database: concert_singer
Question: What is the average, minimum, and maximum age of all singers from France?
Gold SQL: SELECT avg(age) ,  min(age) ,  max(age) FROM singer WHERE country  =  'France'
Fine-tuned prediction: SELECT avg(age) ,  min(age) ,  max(age) FROM singer WHERE country  =  "France"

Database: concert_singer
Question: What is the average, minimum, and maximum age for all French singers?
Gold SQL: SELECT avg(age) ,  min(age) ,  max(age) FROM singer WHERE country  =  'France'
Fine-tuned prediction: SELECT avg(age) ,  min(age) ,  max(age) FROM singer WHERE country  =  "France"

Database: concert_singer
Question: Show the name and the release year of the song by the youngest singer.
Gold SQL: SELECT song_name ,  song_release_year FROM singer ORDER BY age LIMIT 1
Fine-tuned prediction: SELECT Song_Name ,  Song_release_year FROM singer WHERE Age  =  ( SELECT MIN ( Age ) FROM singer )

Database: concert_singer
Question: What are the n

# STEP 12: Save adapter, tokenizer, metrics, and predictions

In [18]:
# ============================================================
# STEP 12: Save adapter, tokenizer, metrics, and predictions
# ============================================================

import json
import shutil

from pathlib import Path


# ------------------------------------------------------------
# 1. Create the project directories
# ------------------------------------------------------------

PROJECT_DIR = Path(
    "/kaggle/working/qwen3-4b-spider-unsloth"
)

ADAPTER_DIR = PROJECT_DIR / "final_adapter"
METRICS_DIR = PROJECT_DIR / "metrics"
PREDICTIONS_DIR = PROJECT_DIR / "predictions"

ADAPTER_DIR.mkdir(parents=True, exist_ok=True)
METRICS_DIR.mkdir(parents=True, exist_ok=True)
PREDICTIONS_DIR.mkdir(parents=True, exist_ok=True)

print("Project directory:", PROJECT_DIR)


# ------------------------------------------------------------
# 2. Save the trained LoRA adapter and tokenizer
# ------------------------------------------------------------

model.save_pretrained(
    ADAPTER_DIR,
    safe_serialization=True,
)

tokenizer.save_pretrained(
    ADAPTER_DIR,
)

print("\nAdapter and tokenizer saved successfully")


# ------------------------------------------------------------
# 3. Save the evaluation comparison
# ------------------------------------------------------------

comparison_path = (
    METRICS_DIR / "base_vs_finetuned_comparison.csv"
)

comparison_df.to_csv(
    comparison_path,
    index=False,
)

print("Comparison saved to:", comparison_path)


# ------------------------------------------------------------
# 4. Copy existing training and evaluation artifacts
# ------------------------------------------------------------

artifacts_to_copy = {
    "/kaggle/working/training_history.csv":
        METRICS_DIR / "training_history.csv",

    "/kaggle/working/training_summary.json":
        METRICS_DIR / "training_summary.json",

    "/kaggle/working/finetuned_evaluation_summary.json":
        METRICS_DIR / "finetuned_evaluation_summary.json",

    "/kaggle/working/base_model_predictions.csv":
        PREDICTIONS_DIR / "base_model_predictions.csv",

    "/kaggle/working/finetuned_model_predictions.csv":
        PREDICTIONS_DIR / "finetuned_model_predictions.csv",
}

for source_path, destination_path in artifacts_to_copy.items():

    source = Path(source_path)

    if source.exists():
        shutil.copy2(
            source,
            destination_path,
        )

        print(
            "Copied:",
            destination_path.relative_to(PROJECT_DIR),
        )

    else:
        print(
            "Warning - file not found:",
            source_path,
        )


# ------------------------------------------------------------
# 5. Create the final project summary
# ------------------------------------------------------------

project_summary = {
    "project_name": "Qwen3-4B Text-to-SQL Fine-Tuning with Unsloth",
    "task": "Schema-aware natural-language to SQLite generation",

    "base_model": "unsloth/Qwen3-4B-unsloth-bnb-4bit",
    "dataset": "hujudev/spider-text-2-sql",

    "framework": "Unsloth",
    "training_method": "4-bit QLoRA",

    "hardware": {
        "platform": "Kaggle",
        "gpu": "Tesla T4",
        "visible_gpu_count": 1,
        "gpu_memory_gb": 14.56,
        "precision": "FP16",
    },

    "dataset_statistics": {
        "training_examples_after_deduplication": 8651,
        "overlength_training_examples_removed": 96,
        "final_training_examples": 8555,
        "validation_examples": 1034,
        "train_validation_database_overlap": 0,
    },

    "lora_configuration": {
        "rank": 16,
        "alpha": 32,
        "dropout": 0,
        "trainable_parameters": 33030144,
        "trainable_percentage": 0.81,
        "base_model_frozen": True,
    },

    "training_configuration": {
        "steps": 100,
        "effective_batch_size": 8,
        "approximate_sequences_processed": 800,
        "learning_rate": 0.0002,
        "optimizer": "adamw_8bit",
        "maximum_sequence_length": 2048,
        "response_only_training": True,
        "training_minutes": 18.15,
        "peak_gpu_memory_gb": 4.77,
        "average_training_loss": 0.40030657112598417,
        "final_logged_loss": 0.208128,
    },

    "base_model_evaluation": {
        "sql_syntax_validity_percent": 99.61,
        "sql_only_format_compliance_percent": 99.71,
        "normalized_exact_match_percent": 22.63,
        "evaluation_minutes": 28.86,
    },

    "finetuned_model_evaluation": {
        "sql_syntax_validity_percent": 100.00,
        "sql_only_format_compliance_percent": 100.00,
        "normalized_exact_match_percent": 39.46,
        "evaluation_minutes": 29.31,
    },

    "improvement": {
        "normalized_exact_match_percentage_points": 16.83,
        "normalized_exact_match_relative_percent": 74.4,
    },

    "evaluation_note": (
        "Normalized exact match is a strict structural metric. "
        "It may mark semantically equivalent SQL queries as different."
    ),
}

summary_path = PROJECT_DIR / "project_summary.json"

with open(
    summary_path,
    "w",
    encoding="utf-8",
) as file:

    json.dump(
        project_summary,
        file,
        indent=2,
    )

print("\nProject summary saved to:", summary_path)


# ------------------------------------------------------------
# 6. Verify that this is an adapter-only artifact
# ------------------------------------------------------------

adapter_config_path = (
    ADAPTER_DIR / "adapter_config.json"
)

adapter_weight_candidates = [
    ADAPTER_DIR / "adapter_model.safetensors",
    ADAPTER_DIR / "adapter_model.bin",
]

assert adapter_config_path.exists(), (
    "adapter_config.json was not saved."
)

assert any(
    path.exists()
    for path in adapter_weight_candidates
), "Adapter weights were not saved."

unexpected_full_model_files = [
    ADAPTER_DIR / "model.safetensors",
    ADAPTER_DIR / "pytorch_model.bin",
]

unexpected_files = [
    path.name
    for path in unexpected_full_model_files
    if path.exists()
]

assert not unexpected_files, (
    "Unexpected full-model files detected: "
    f"{unexpected_files}"
)

print("\nAdapter-only verification: PASSED")


# ------------------------------------------------------------
# 7. Display the saved files
# ------------------------------------------------------------

print("\nSaved project files")
print("=" * 60)

for file_path in sorted(PROJECT_DIR.rglob("*")):

    if file_path.is_file():

        file_size_mb = (
            file_path.stat().st_size / (1024 ** 2)
        )

        relative_path = file_path.relative_to(
            PROJECT_DIR
        )

        print(
            f"{relative_path} "
            f"({file_size_mb:.2f} MB)"
        )


# ------------------------------------------------------------
# 8. Create a downloadable ZIP archive
# ------------------------------------------------------------

ZIP_BASE_PATH = Path(
    "/kaggle/working/qwen3-4b-spider-unsloth-artifacts"
)

zip_path = Path(
    shutil.make_archive(
        str(ZIP_BASE_PATH),
        "zip",
        root_dir=PROJECT_DIR.parent,
        base_dir=PROJECT_DIR.name,
    )
)

assert zip_path.exists(), (
    "ZIP archive creation failed."
)

zip_size_mb = (
    zip_path.stat().st_size / (1024 ** 2)
)

print("\nProject ZIP created successfully")
print("ZIP path:", zip_path)
print(f"ZIP size: {zip_size_mb:.2f} MB")
print("\nStep 12 artifact verification: PASSED")

Project directory: /kaggle/working/qwen3-4b-spider-unsloth


Unsloth: Restored added_tokens_decoder metadata in /kaggle/working/qwen3-4b-spider-unsloth/final_adapter/tokenizer_config.json.



Adapter and tokenizer saved successfully
Comparison saved to: /kaggle/working/qwen3-4b-spider-unsloth/metrics/base_vs_finetuned_comparison.csv
Copied: metrics/training_history.csv
Copied: metrics/training_summary.json
Copied: metrics/finetuned_evaluation_summary.json
Copied: predictions/base_model_predictions.csv
Copied: predictions/finetuned_model_predictions.csv

Project summary saved to: /kaggle/working/qwen3-4b-spider-unsloth/project_summary.json

Adapter-only verification: PASSED

Saved project files
final_adapter/README.md (0.00 MB)
final_adapter/adapter_config.json (0.00 MB)
final_adapter/adapter_model.safetensors (126.06 MB)
final_adapter/chat_template.jinja (0.00 MB)
final_adapter/tokenizer.json (10.89 MB)
final_adapter/tokenizer_config.json (0.00 MB)
metrics/base_vs_finetuned_comparison.csv (0.00 MB)
metrics/finetuned_evaluation_summary.json (0.00 MB)
metrics/training_history.csv (0.00 MB)
metrics/training_summary.json (0.00 MB)
predictions/base_model_predictions.csv (0.55 M

# STEP 13: Create model card and upload adapter to Hugging Face


In [25]:
# ============================================================
# STEP 13: Publish the adapter to Hugging Face
# ============================================================

import shutil

from pathlib import Path
from kaggle_secrets import UserSecretsClient
from huggingface_hub import HfApi, login


# ------------------------------------------------------------
# 1. Paths
# ------------------------------------------------------------

PROJECT_DIR = Path(
    "/kaggle/working/qwen3-4b-spider-unsloth"
)

ADAPTER_DIR = PROJECT_DIR / "final_adapter"
METRICS_DIR = PROJECT_DIR / "metrics"
RESULTS_DIR = ADAPTER_DIR / "results"

REPOSITORY_NAME = "qwen3-4b-spider-unsloth"

assert ADAPTER_DIR.exists()
assert (ADAPTER_DIR / "adapter_config.json").exists()
assert (ADAPTER_DIR / "adapter_model.safetensors").exists()

print("Adapter verification: PASSED")


# ------------------------------------------------------------
# 2. Authenticate with Hugging Face
# ------------------------------------------------------------

secrets = UserSecretsClient()

try:
    HF_TOKEN = secrets.get_secret("HF_TOKEN")

except Exception as error:
    raise RuntimeError(
        "HF_TOKEN was not found. Add and enable it in "
        "Kaggle Add-ons -> Secrets."
    ) from error

assert HF_TOKEN, "HF_TOKEN is empty."

login(
    token=HF_TOKEN,
    add_to_git_credential=False,
)

api = HfApi()

user_information = api.whoami(
    token=HF_TOKEN
)

HF_USERNAME = user_information["name"]
REPO_ID = f"{HF_USERNAME}/{REPOSITORY_NAME}"

print("Authenticated user:", HF_USERNAME)
print("Target repository:", REPO_ID)


# ------------------------------------------------------------
# 3. Add summarized result files
# ------------------------------------------------------------

RESULTS_DIR.mkdir(
    parents=True,
    exist_ok=True,
)

result_files = [
    "base_vs_finetuned_comparison.csv",
    "training_history.csv",
    "training_summary.json",
    "finetuned_evaluation_summary.json",
]

for file_name in result_files:

    source = METRICS_DIR / file_name
    destination = RESULTS_DIR / file_name

    if source.exists():
        shutil.copy2(source, destination)
        print("Added:", file_name)

project_summary = PROJECT_DIR / "project_summary.json"

if project_summary.exists():

    shutil.copy2(
        project_summary,
        RESULTS_DIR / "project_summary.json",
    )

    print("Added: project_summary.json")


# ------------------------------------------------------------
# 4. Create a concise professional model card
# ------------------------------------------------------------

model_card_lines = [
    "---",
    "base_model: unsloth/Qwen3-4B-unsloth-bnb-4bit",
    "library_name: peft",
    "pipeline_tag: text-generation",
    "datasets:",
    "- hujudev/spider-text-2-sql",
    "language:",
    "- en",
    "tags:",
    "- qwen3",
    "- unsloth",
    "- qlora",
    "- peft",
    "- text-to-sql",
    "- spider",
    "- sqlite",
    "---",
    "",
    "# Qwen3-4B Spider Text-to-SQL - Unsloth QLoRA",
    "",
    "A text-to-SQL adapter fine-tuned from",
    "`unsloth/Qwen3-4B-unsloth-bnb-4bit` using",
    "Unsloth-accelerated 4-bit QLoRA.",
    "",
    "The model accepts a SQLite database schema and a",
    "natural-language question and generates a SQL query.",
    "",
    "## Evaluation Results",
    "",
    "Evaluation used all 1,034 Spider validation examples.",
    "Training and validation databases were completely separated.",
    "",
    "| Metric | Base | Fine-tuned | Change |",
    "|---|---:|---:|---:|",
    "| Normalized exact match | 22.63% | **39.46%** | **+16.83 points** |",
    "| SQL syntax validity | 99.61% | **100.00%** | +0.39 points |",
    "| SQL-only compliance | 99.71% | **100.00%** | +0.29 points |",
    "",
    "Normalized exact match improved by approximately",
    "**74.4% relative** over the untouched base model.",
    "",
    "The reported exact match is a strict SQLGlot-normalized",
    "metric, not execution accuracy. Semantically equivalent",
    "queries may still be counted as different.",
    "",
    "## Training Configuration",
    "",
    "| Setting | Value |",
    "|---|---|",
    "| Framework | Unsloth |",
    "| Method | 4-bit QLoRA |",
    "| LoRA rank / alpha | 16 / 32 |",
    "| Trainable parameters | 33,030,144 (0.81%) |",
    "| Training examples | 8,555 |",
    "| Validation examples | 1,034 |",
    "| Maximum sequence length | 2,048 |",
    "| Training steps | 100 |",
    "| Effective batch size | 8 |",
    "| Learning rate | 2e-4 |",
    "| Optimizer | AdamW 8-bit |",
    "| Precision | FP16 |",
    "| Training time | 18.15 minutes |",
    "| Peak training GPU memory | 4.77 GB |",
    "| Hardware | Single Tesla T4 |",
    "",
    "This was a compact 100-step portfolio experiment and",
    "did not complete a full training epoch.",
    "",
    "## Dataset Preparation",
    "",
    "- Database-level train-validation separation",
    "- Zero overlapping databases between splits",
    "- Missing-value and duplicate validation",
    "- Eight exact training duplicates removed",
    "- Qwen3 conversational formatting",
    "- Response-only assistant training",
    "- Token-length auditing",
    "- 96 overlength training examples removed",
    "",
    "## Example",
    "",
    "**Input**",
    "",
    "```text",
    "Schema: CREATE TABLE singer (name TEXT, age INTEGER);",
    "Question: How many singers are older than 30?",
    "```",
    "",
    "**Output**",
    "",
    "```sql",
    "SELECT COUNT(*) FROM singer WHERE age > 30",
    "```",
    "",
    "## Intended Use",
    "",
    "- Text-to-SQL demonstrations",
    "- SQLite query-generation experiments",
    "- Parameter-efficient fine-tuning examples",
    "- Unsloth and QLoRA workflow demonstrations",
    "",
    "## Limitations",
    "",
    "- The adapter was trained for only 100 steps.",
    "- Evaluation used normalized exact match, not execution accuracy.",
    "- Syntactically valid SQL may still be logically incorrect.",
    "- Performance outside English Spider-style SQLite tasks is unknown.",
    "- Queries should be validated before database execution.",
    "",
    "## Artifacts",
    "",
    "The `results` directory contains the training history,",
    "evaluation summary, comparison metrics, and project summary.",
    "",
    "## Acknowledgements",
    "",
    "Built with Qwen3, Unsloth, Hugging Face PEFT, TRL,",
    "the Spider benchmark, and SQLGlot.",
]

MODEL_CARD = "\n".join(model_card_lines)

model_card_path = ADAPTER_DIR / "README.md"

model_card_path.write_text(
    MODEL_CARD,
    encoding="utf-8",
)

assert model_card_path.exists()
assert model_card_path.stat().st_size > 1000

print("\nModel card created:")
print(model_card_path)


# ------------------------------------------------------------
# 5. Create the public model repository
# ------------------------------------------------------------

repository_url = api.create_repo(
    repo_id=REPO_ID,
    repo_type="model",
    private=False,
    exist_ok=True,
    token=HF_TOKEN,
)

print("\nRepository created:")
print(repository_url)


# ------------------------------------------------------------
# 6. Upload the adapter
# ------------------------------------------------------------

upload_result = api.upload_folder(
    repo_id=REPO_ID,
    repo_type="model",
    folder_path=str(ADAPTER_DIR),
    commit_message=(
        "Upload Unsloth QLoRA adapter and evaluation results"
    ),
    token=HF_TOKEN,
)

print("\nUpload completed:")
print(upload_result)


# ------------------------------------------------------------
# 7. Final verification
# ------------------------------------------------------------

MODEL_URL = f"https://huggingface.co/{REPO_ID}"

print("\n" + "=" * 60)
print("Hugging Face publication completed")
print("=" * 60)
print("Repository:", REPO_ID)
print("Public URL:", MODEL_URL)
print("Full base model uploaded: NO")
print("LoRA adapter uploaded: YES")
print("Tokenizer uploaded: YES")
print("Model card uploaded: YES")
print("Evaluation summaries uploaded: YES")
print("\nStep 13 publication verification: PASSED")

Adapter verification: PASSED
Authenticated user: shoron07
Target repository: shoron07/qwen3-4b-spider-unsloth
Added: base_vs_finetuned_comparison.csv
Added: training_history.csv
Added: training_summary.json
Added: finetuned_evaluation_summary.json
Added: project_summary.json

Model card created:
/kaggle/working/qwen3-4b-spider-unsloth/final_adapter/README.md

Repository created:
https://huggingface.co/shoron07/qwen3-4b-spider-unsloth


Processing Files (0 / 0): |          |  0.00B /  0.00B            

New Data Upload: |          |  0.00B /  0.00B            


Upload completed:
https://huggingface.co/shoron07/qwen3-4b-spider-unsloth/commit/3955ea6de28ad5079659d6fece37e66a734ca67f

Hugging Face publication completed
Repository: shoron07/qwen3-4b-spider-unsloth
Public URL: https://huggingface.co/shoron07/qwen3-4b-spider-unsloth
Full base model uploaded: NO
LoRA adapter uploaded: YES
Tokenizer uploaded: YES
Model card uploaded: YES
Evaluation summaries uploaded: YES

Step 13 publication verification: PASSED


In [24]:
from kaggle_secrets import UserSecretsClient

HF_TOKEN = UserSecretsClient().get_secret("HF_TOKEN")

assert HF_TOKEN, "HF_TOKEN is empty."

print("HF_TOKEN access: PASSED")

HF_TOKEN access: PASSED


In [26]:
from IPython.display import FileLink, display

display(
    FileLink(
        "/kaggle/working/qwen3-4b-spider-unsloth-artifacts.zip"
    )
)


/kaggle/working/qwen3-4b-spider-unsloth-artifacts.zip

In [28]:
import os

from pathlib import Path
from IPython.display import FileLink, display

zip_name = "qwen3-4b-spider-unsloth-artifacts.zip"
zip_path = Path("/kaggle/working") / zip_name

assert zip_path.is_file(), f"File not found: {zip_path}"

print(
    f"ZIP verified: "
    f"{zip_path.stat().st_size / (1024 ** 2):.2f} MB"
)

# FileLink works correctly with a relative path
os.chdir("/kaggle/working")

display(
    FileLink(zip_name)
)

ZIP verified: 118.86 MB


/kaggle/working/qwen3-4b-spider-unsloth-artifacts.zip